# Soma de Vetores em CUDA — Benchmark CPU vs GPU

**Disciplina:** Arquitetura de Computadores  
**Aluno:** Wésio Filho  
**Professor:** Newarney T. da Costa

Este notebook executa o benchmark do projeto usando os arquivos reais do repositório:

- `codigo/soma_cpu.c`
- `codigo/soma_gpu.cu`

Ele não reescreve versões simplificadas do código. Assim, o que é medido aqui é o mesmo código entregue no GitHub.


## 1. Verificar GPU ativa

Este notebook foi preparado para execução **local/bare metal** em Linux com GPU NVIDIA.

Antes de rodar tudo, confira se o driver NVIDIA está ativo com `nvidia-smi`.


In [ ]:
!nvidia-smi

## 2. Preparar o repositório

Esta célula encontra a raiz do repositório local. Ela funciona quando o notebook é aberto:

- pela raiz do projeto; ou
- de dentro da pasta `notebook/`.

O notebook **não reescreve** os códigos C/CUDA. Ele usa os arquivos reais do repositório.


In [ ]:
from pathlib import Path
import os


def encontrar_raiz_repo():
    atual = Path.cwd().resolve()
    candidatos = [atual, *atual.parents]
    for candidato in candidatos:
        if (candidato / 'codigo' / 'soma_cpu.c').exists() and (candidato / 'codigo' / 'soma_gpu.cu').exists():
            return candidato
    raise FileNotFoundError('Não encontrei codigo/soma_cpu.c e codigo/soma_gpu.cu a partir do diretório atual.')

RAIZ_REPO = encontrar_raiz_repo()
os.chdir(RAIZ_REPO)

print('Diretório atual:', Path.cwd())
print('Arquivos de código:')
print(' -', Path('codigo/soma_cpu.c').resolve())
print(' -', Path('codigo/soma_gpu.cu').resolve())


## 3. Compilar os programas

A versão CPU é compilada com `gcc` e a versão GPU com `nvcc`.


In [ ]:
!gcc -std=c11 -O2 -Wall -Wextra codigo/soma_cpu.c -o soma_cpu
!nvcc -O2 codigo/soma_gpu.cu -o soma_gpu

## 4. Conferir os primeiros resultados

Aqui rodamos com `N = 10` em modo normal, sem `--benchmark`, para mostrar os primeiros valores de `C`.

Como `A[i] = i` e `B[i] = 2*i`, então `C[i] = 3*i`.


In [ ]:
!./soma_cpu 10
!./soma_gpu 10

## 5. Benchmark

Agora rodamos cada versão 5 vezes para cada tamanho de vetor exigido no enunciado:

- 1.000 elementos
- 100.000 elementos
- 1.000.000 elementos
- 10.000.000 elementos

Os programas são executados com `--benchmark`, então a saída fica no formato:

```text
CPU 3.421000
GPU 0.842000
```

> Observação: se a GPU estiver ocupada por outro processo pesado (por exemplo, um servidor local de IA usando VRAM), feche esse processo antes de rodar o benchmark completo.


In [ ]:
import csv
import re
import subprocess
from pathlib import Path

TAMANHOS_N = [1000, 100000, 1000000, 10000000]
NUM_RUNS = 5
RESULTADOS_DIR = Path('resultados')
RESULTADOS_DIR.mkdir(exist_ok=True)

PADRAO_SAIDA = re.compile(r'^(CPU|GPU)\s+([0-9]+(?:\.[0-9]+)?)$')


def executar_benchmark(comando, rotulo_esperado):
    resultado = subprocess.run(comando, capture_output=True, text=True)

    if resultado.returncode != 0:
        raise RuntimeError(
            'Falha ao executar: ' + ' '.join(comando) + '\n'
            + 'STDOUT:\n' + resultado.stdout + '\n'
            + 'STDERR:\n' + resultado.stderr
        )

    saida = resultado.stdout.strip()
    match = PADRAO_SAIDA.match(saida)
    if not match:
        raise ValueError(
            'Saída inesperada do comando: ' + ' '.join(comando) + '\n'
            + 'Saída recebida:\n' + saida
        )

    rotulo, tempo = match.groups()
    if rotulo != rotulo_esperado:
        raise ValueError(f'Esperado {rotulo_esperado}, mas recebi {rotulo}')
    return float(tempo)

linhas = []

for N in TAMANHOS_N:
    for run in range(1, NUM_RUNS + 1):
        print(f'Rodando N={N}, execução {run}/{NUM_RUNS}...')
        tempo_cpu = executar_benchmark(['./soma_cpu', str(N), '--benchmark'], 'CPU')
        tempo_gpu = executar_benchmark(['./soma_gpu', str(N), '--benchmark'], 'GPU')
        linhas.append({
            'N': N,
            'run': run,
            'tempo_cpu_ms': tempo_cpu,
            'tempo_gpu_ms': tempo_gpu,
            'speedup': tempo_cpu / tempo_gpu,
        })

with open(RESULTADOS_DIR / 'tempos.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['N', 'run', 'tempo_cpu_ms', 'tempo_gpu_ms', 'speedup'])
    writer.writeheader()
    writer.writerows(linhas)

print('Arquivo salvo:', RESULTADOS_DIR / 'tempos.csv')
for linha in linhas[:5]:
    print(linha)


## 6. Calcular médias e speedup

A tabela abaixo resume a média das 5 execuções por tamanho de vetor.


In [ ]:
import csv
import statistics
from collections import defaultdict

por_n = defaultdict(list)
for linha in linhas:
    por_n[linha['N']].append(linha)

medias = []
for N in sorted(por_n):
    grupo = por_n[N]
    cpu = [x['tempo_cpu_ms'] for x in grupo]
    gpu = [x['tempo_gpu_ms'] for x in grupo]
    media_cpu = statistics.mean(cpu)
    media_gpu = statistics.mean(gpu)
    medias.append({
        'N': N,
        'tempo_cpu_ms_medio': media_cpu,
        'tempo_cpu_ms_desvio': statistics.stdev(cpu) if len(cpu) > 1 else 0.0,
        'tempo_gpu_ms_medio': media_gpu,
        'tempo_gpu_ms_desvio': statistics.stdev(gpu) if len(gpu) > 1 else 0.0,
        'speedup_medio': media_cpu / media_gpu,
    })

with open(RESULTADOS_DIR / 'tempos_medios.csv', 'w', newline='', encoding='utf-8') as f:
    campos = ['N', 'tempo_cpu_ms_medio', 'tempo_cpu_ms_desvio', 'tempo_gpu_ms_medio', 'tempo_gpu_ms_desvio', 'speedup_medio']
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()
    writer.writerows(medias)

print('Arquivo salvo:', RESULTADOS_DIR / 'tempos_medios.csv')
print('| N | CPU médio (ms) | GPU médio (ms) | Speedup |')
print('|---:|---:|---:|---:|')
for m in medias:
    print(f"| {m['N']} | {m['tempo_cpu_ms_medio']:.6f} | {m['tempo_gpu_ms_medio']:.6f} | {m['speedup_medio']:.2f}x |")


## 7. Gráfico 1 — Tempo CPU vs GPU

O eixo Y usa escala logarítmica para facilitar a comparação entre valores pequenos e grandes.


In [ ]:
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

Ns = [m['N'] for m in medias]
tempos_cpu = [m['tempo_cpu_ms_medio'] for m in medias]
tempos_gpu = [m['tempo_gpu_ms_medio'] for m in medias]

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(Ns, tempos_cpu, marker='o', linewidth=2.5, markersize=8, label='CPU sequencial', color='#e74c3c')
ax.plot(Ns, tempos_gpu, marker='s', linewidth=2.5, markersize=8, label='GPU CUDA', color='#2ecc71')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tamanho do vetor (N)', fontsize=12)
ax.set_ylabel('Tempo médio (ms)', fontsize=12)
ax.set_title('Tempo de execução: CPU vs GPU', fontsize=14, fontweight='bold')
ax.set_xticks(Ns)
ax.set_xticklabels([f'{n:,}'.replace(',', '.') for n in Ns])
ax.legend(fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / 'grafico_tempos.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Gráfico 2 — Speedup

O speedup mostra quantas vezes a versão GPU foi mais rápida que a versão CPU:

```text
speedup = tempo_CPU / tempo_GPU
```


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

speedups = [m['speedup_medio'] for m in medias]
cores = ['#e74c3c' if s < 1 else '#2ecc71' for s in speedups]
barras = ax.bar([str(n) for n in Ns], speedups, color=cores, edgecolor='white', linewidth=1.5)

ax.axhline(y=1, color='gray', linestyle='--', linewidth=1.2, label='Speedup = 1 (empate)')

for barra, valor in zip(barras, speedups):
    altura = barra.get_height()
    ax.text(barra.get_x() + barra.get_width() / 2, altura, f'{valor:.2f}x', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel('Tamanho do vetor (N)', fontsize=12)
ax.set_ylabel('Speedup médio (CPU/GPU)', fontsize=12)
ax.set_title('Speedup da GPU em relação à CPU', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / 'grafico_speedup.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Arquivos gerados

Ao final da execução, os principais arquivos ficam em `resultados/`:

- `resultados/tempos.csv`
- `resultados/tempos_medios.csv`
- `resultados/grafico_tempos.png`
- `resultados/grafico_speedup.png`


In [ ]:
print('Arquivos em resultados/:')
for arquivo in sorted(RESULTADOS_DIR.iterdir()):
    print('-', arquivo)


---

## Conclusão esperada

Para vetores pequenos, a CPU pode vencer porque a GPU tem overhead de transferência de dados e lançamento da kernel. Para vetores grandes, a GPU tende a vencer porque milhares de threads trabalham em paralelo, como um exército de clones fazendo a mesma missão em partes diferentes do vetor.
